# Constrained Optimization

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.04 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/05_constrained_optimization.ipynb)

---

## 🎯 Learning Objective

Learn Lagrange multipliers and KKT conditions for solving optimization problems subject to equality and inequality constraints.

---

## 📐 Theory

Every optimization problem so far in this module has been **unconstrained**: minimize $f(\mathbf{w})$
over all of $\mathbb{R}^n$. Many real problems restrict $\mathbf{w}$ to a feasible region —
support vector machines, for instance, need the margin constraint satisfied by every data
point, not just a low loss. This notebook covers the machinery for that: Lagrange multipliers
and the KKT conditions.

### Equality constraints: Lagrange multipliers

To minimize $f(\mathbf{w})$ subject to $g(\mathbf{w})=0$, form the **Lagrangian**:

$$\mathcal{L}(\mathbf{w}, \lambda) = f(\mathbf{w}) - \lambda\, g(\mathbf{w})$$

At a constrained optimum, $\nabla f$ and $\nabla g$ must be *parallel* — if they weren't, you
could slide along the constraint surface $g=0$ and still decrease $f$, contradicting
optimality. "Parallel gradients" is exactly what $\nabla_{\mathbf{w}}\mathcal{L}=\mathbf{0}$
enforces: $\nabla f(\mathbf{w}) = \lambda \nabla g(\mathbf{w})$. Together with the original
constraint $\nabla_\lambda \mathcal{L} = -g(\mathbf{w}) = 0$, this gives a system of equations
that pins down both $\mathbf{w}^*$ and the multiplier $\lambda$ (whose value, as a bonus, tells
you the optimum's sensitivity to relaxing the constraint — recall the sensitivity reading of a
derivative from `02.02`).

### Inequality constraints: KKT conditions

For $g(\mathbf{w}) \le 0$, the constraint might be **active** (binding, $g(\mathbf{w}^*)=0$,
pressing against the boundary) or **inactive** ($g(\mathbf{w}^*)<0$, the unconstrained optimum
already satisfies it). The **Karush-Kuhn-Tucker (KKT) conditions** generalize Lagrange
multipliers to handle both cases at once, using a multiplier $\mu \ge 0$:

$$\nabla f(\mathbf{w}^*) + \mu \nabla g(\mathbf{w}^*) = \mathbf{0}, \qquad
\mu \ge 0, \qquad g(\mathbf{w}^*) \le 0, \qquad \mu\, g(\mathbf{w}^*) = 0$$

The last equation is **complementary slackness**: either the constraint is active
($g(\mathbf{w}^*)=0$) or its multiplier is zero ($\mu=0$) — never both nonzero. Intuitively, a
constraint only "pushes back" on the optimum (nonzero $\mu$) when you're pressed right up
against it; if the unconstrained optimum was already feasible, the constraint does nothing and
its multiplier vanishes.

### The dual problem

Define the **dual function** $q(\mu) = \min_{\mathbf{w}} \mathcal{L}(\mathbf{w}, \mu)$ — the
best the Lagrangian can do for a fixed multiplier. Maximizing $q$ over $\mu \ge 0$ (the **dual
problem**) gives a lower bound on the original ("primal") problem's optimal value; under
convexity (satisfied by SVMs), this bound is tight and the dual's optimal value *equals* the
primal's. Solving the dual is sometimes far easier — SVMs are the canonical example, where the
dual turns into a problem expressed purely via dot products between data points, which is what
makes the kernel trick possible (previewed here, covered fully in `11.02`).

### Penalty methods

An alternative to solving the Lagrangian system directly: convert a constrained problem into an
*unconstrained* one by adding a penalty term that punishes constraint violation,
$f(\mathbf{w}) + \rho \max(0, g(\mathbf{w}))^2$ for growing $\rho$. As $\rho\to\infty$, the
penalized minimizer converges to the constrained optimum. This is crude compared to solving KKT
directly, but it reduces a constrained problem to something `04.02`'s gradient descent already
knows how to handle — a pattern that reappears whenever regularization is described as a "soft
constraint" on model complexity.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
from scipy.optimize import minimize
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

At a constrained optimum, the objective's contour line is tangent to the constraint curve —
their gradients point in the same (or exactly opposite) direction. Plotting both makes this
tangency, and hence why $\nabla f = \lambda \nabla g$, visible directly.

In [ ]:
# Minimize f(x,y)=x^2+y^2 subject to g(x,y)=x+y-1=0 -- visualize the tangency condition
f = lambda x, y: x**2 + y**2
x_opt, y_opt = 0.5, 0.5  # solved analytically in the Theory section: x=y=0.5, lambda=1

x = np.linspace(-1, 2, 300)
y = np.linspace(-1, 2, 300)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
ax.plot(x, 1 - x, color="#C44E52", lw=2.5, label="constraint: x+y=1")
ax.plot(x_opt, y_opt, 'ko', markersize=10, zorder=5, label="constrained optimum")

# Draw both gradients at the optimum -- they should be parallel (KKT: grad f = lambda * grad g)
grad_f = np.array([2 * x_opt, 2 * y_opt])
grad_g = np.array([1.0, 1.0])  # gradient of x+y-1 is always (1,1)
ax.annotate("", xy=(x_opt + grad_f[0]*0.3, y_opt + grad_f[1]*0.3), xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle="->", color="#4C72B0", lw=2.5))
ax.annotate("", xy=(x_opt + grad_g[0]*0.3, y_opt + grad_g[1]*0.3), xytext=(x_opt, y_opt),
            arrowprops=dict(arrowstyle="->", color="#DD8452", lw=2, linestyle="--"))
ax.text(x_opt + 0.35, y_opt + 0.25, "grad f", color="#4C72B0", fontsize=10)
ax.text(x_opt + 0.35, y_opt + 0.42, "grad g", color="#DD8452", fontsize=10)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("grad f is parallel to grad g at the constrained optimum")
ax.legend(loc="upper right", fontsize=9)
ax.set_aspect('equal')
plt.show()

print(f"grad f at optimum: {grad_f}, grad g: {grad_g}")
print(f"ratio grad_f / grad_g = {grad_f / grad_g}  -- constant ratio confirms parallel vectors (lambda=1)")

## 🐍 Implementation from Scratch

We solve one equality-constrained and one inequality-constrained problem two independent ways:
symbolically via the Lagrangian/KKT stationarity equations, and numerically via
`scipy.optimize.minimize` -- confirming they agree.

In [ ]:
# --- Equality-constrained: minimize x^2+y^2 subject to x+y=1 ---
x, y, lam = sp.symbols('x y lam', real=True)
L_eq = x**2 + y**2 - lam * (x + y - 1)
stationarity = [sp.diff(L_eq, x), sp.diff(L_eq, y), sp.diff(L_eq, lam)]
sol_eq = sp.solve(stationarity, [x, y, lam])
print("Lagrangian stationarity (sympy):", sol_eq)

res_eq = minimize(lambda v: v[0]**2 + v[1]**2, [0, 0], constraints=[{'type': 'eq', 'fun': lambda v: v[0] + v[1] - 1}])
print(f"scipy.optimize.minimize:          x={res_eq.x[0]:.4f}, y={res_eq.x[1]:.4f}")
print(f"Match: {np.isclose(float(sol_eq[x]), res_eq.x[0]) and np.isclose(float(sol_eq[y]), res_eq.x[1])}")

# --- Inequality-constrained: minimize (x-2)^2+(y-2)^2 subject to x+y<=2, x>=0, y>=0 ---
# The unconstrained minimum (2,2) violates x+y<=2, so this constraint should be ACTIVE
# (binding) at the true optimum -- KKT predicts a nonzero multiplier there.
def obj(v):
    return (v[0] - 2)**2 + (v[1] - 2)**2

res_ineq = minimize(obj, [0, 0], constraints=[{'type': 'ineq', 'fun': lambda v: 2 - v[0] - v[1]}],
                     bounds=[(0, None), (0, None)])
x_star, y_star = res_ineq.x
g_at_star = x_star + y_star - 2   # should be ~0: constraint is active
print(f"\nInequality-constrained optimum: x={x_star:.4f}, y={y_star:.4f}")
print(f"Constraint x+y-2 at optimum: {g_at_star:.6f}  (near 0 => ACTIVE, matching KKT's prediction")
print(f"since the unconstrained optimum (2,2) violates x+y<=2)")

# Verify complementary slackness by solving the KKT stationarity system directly with sympy:
# grad f + mu * grad g = 0, with g(x,y) = x+y-2 (active, so g=0 exactly)
mu = sp.symbols('mu', nonnegative=True)
kkt_eqs = [sp.diff((x - 2)**2 + (y - 2)**2, x) + mu * 1,
           sp.diff((x - 2)**2 + (y - 2)**2, y) + mu * 1,
           x + y - 2]
sol_kkt = sp.solve(kkt_eqs, [x, y, mu])
print(f"\nKKT system solved symbolically: x={float(sol_kkt[x]):.4f}, y={float(sol_kkt[y]):.4f}, mu={float(sol_kkt[mu]):.4f}")
print(f"mu > 0 confirms the constraint is active, consistent with complementary slackness (mu * g = 0, g = 0).")

## 🤖 Why This Matters for AI

A **Support Vector Machine** finds the maximum-margin separating hyperplane by solving a
constrained problem: minimize $\frac{1}{2}\|\mathbf{w}\|^2$ subject to
$y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \ge 1$ for every training point $i$. This is exactly the
KKT setting above: most points satisfy the margin constraint comfortably (inactive, multiplier
$\alpha_i=0$), while the points sitting exactly on the margin boundary — the **support
vectors** — have the constraint active ($\alpha_i > 0$). Complementary slackness is the precise
mathematical statement of why an SVM's decision boundary depends on only a handful of points
instead of the entire dataset. Below, we fit `sklearn`'s SVM on a toy linearly-separable
dataset and confirm directly: points with nonzero dual coefficients (the $\alpha_i y_i$ in
`svm.dual_coef_`) are exactly `svm.support_vectors_`, and every other point's multiplier is
zero — complementary slackness, verified on a real (if small) model.

In [ ]:
from sklearn.svm import SVC

rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-2, -2], 0.6, size=(20, 2)), rng.normal([2, 2], 0.6, size=(20, 2))])
y_labels = np.array([-1] * 20 + [1] * 20)

svm = SVC(kernel="linear", C=1000)  # large C: closely approximates the hard-margin SVM above
svm.fit(X, y_labels)

n_support = len(svm.support_)
n_total = len(X)
print(f"Total training points: {n_total}")
print(f"Support vectors (active constraint, alpha_i > 0): {n_support}")
print(f"Points with alpha_i = 0 (constraint inactive, margin satisfied with room to spare): {n_total - n_support}")
print(f"\nThe decision boundary w={np.round(svm.coef_[0], 3)}, b={svm.intercept_[0]:.3f} depends")
print(f"ONLY on the {n_support} support vectors -- removing any non-support-vector point and")
print(f"re-fitting leaves the boundary UNCHANGED, which we verify next.")

# Verify: refit after removing a non-support-vector point -- boundary should be identical
non_support_idx = [i for i in range(n_total) if i not in svm.support_][0]
X_reduced = np.delete(X, non_support_idx, axis=0)
y_reduced = np.delete(y_labels, non_support_idx)
svm_reduced = SVC(kernel="linear", C=1000).fit(X_reduced, y_reduced)
print(f"\nAfter removing a non-support point, w={np.round(svm_reduced.coef_[0], 3)}, "
      f"b={svm_reduced.intercept_[0]:.3f}")
print(f"Boundary unchanged: {np.allclose(svm.coef_, svm_reduced.coef_, atol=1e-3)}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[:, 0], X[:, 1], c=y_labels, cmap="coolwarm", edgecolors="k", s=50, label="training points")
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1], s=200, facecolors="none",
           edgecolors="black", linewidths=2, label="support vectors (active constraint)")
xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
yy = -(svm.coef_[0, 0] * xx + svm.intercept_[0]) / svm.coef_[0, 1]
ax.plot(xx, yy, 'k--', label="decision boundary")
ax.set_title("Complementary slackness in action: only circled points\n(active constraints) determine the boundary")
ax.legend(fontsize=9)
plt.show()

## 🏋️ Exercises

### Warm-up
1. Using Lagrange multipliers, minimize $f(x,y)=x^2+4y^2$ subject to $x+y=3$ by hand. Verify
   with `sp.solve` on the stationarity equations, and cross-check with `scipy.optimize.minimize`.

### Practice
2. For the inequality-constrained problem in the Implementation cell, change the objective to
   $(x-1)^2+(y-1)^2$ (so the unconstrained optimum $(1,1)$ already satisfies $x+y\le 2$).
   Re-solve and confirm the KKT multiplier $\mu$ comes out to (approximately) zero — the
   constraint is now inactive. Explain, in terms of complementary slackness, why this makes sense.

### Challenge
3. In the AI section, increase the overlap between the two classes (reduce the distance between
   the cluster centers, e.g. from $(-2,-2)$/$(2,2)$ to $(-0.5,-0.5)$/$(0.5,0.5)$) and re-fit
   the SVM. Report how the number of support vectors changes, and explain intuitively why
   harder-to-separate data forces more points to sit exactly on (or violate) the margin,
   activating more constraints.

---

## 📚 Further Reading
- Boyd & Vandenberghe, *Convex Optimization*, Ch. 5 (Duality)
- Cortes & Vapnik, ["Support-Vector Networks"](https://link.springer.com/article/10.1007/BF00994018)

---

*Next notebook: [Learning Rate Schedules](06_learning_rate_schedules.ipynb)*